In [1]:
import os
import sys
sys.path.append("../../../../")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import copy
import torch
from datetime import datetime
from utils.helper import ModelConfig, color_print
from utils.dataset_utils.load_dataset import (
    load_data, Embedding
)
from utils.model_utils.save_module import save_module
from utils.model_utils.load_model import load_model
from utils.model_utils.evaluate import evaluate_model, get_sparsity, similar
from utils.dataset_utils.perturb import make_example
from utils.dataset_utils.sampling import SamplingDataset
from utils.prune_utils.prune_head import (
    compute_heads_importance,
    head_importance_prunning,
)
from utils.prune_utils.prune import recover_tangling_identification, prune_concern_identification, prune_wanda

In [3]:
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
from torch.nn import CrossEntropyLoss, MSELoss
from functools import partial
import torch.nn.functional as F
import math
from sklearn.metrics import classification_report
import gc

In [4]:
name= "YahooAnswersTopics"
device = torch.device("cuda:0")
checkpoint = None
batch_size=16
num_workers=4
seed=44

In [5]:
model_config = ModelConfig(name, device)
num_labels = model_config.config["num_labels"]
model, tokenizer, checkpoint = load_model(model_config)

Loading the model.
{'model_name': 'fabriceyhc/bert-base-uncased-yahoo_answers_topics', 'task_type': 'classification', 'architectures': 'bert', 'dataset_name': 'YahooAnswersTopics', 'num_labels': 10, 'cache_dir': 'Models'}
The model fabriceyhc/bert-base-uncased-yahoo_answers_topics is loaded.


{'model_name': 'fabriceyhc/bert-base-uncased-yahoo_answers_topics', 'task_type': 'classification', 'architectures': 'bert', 'dataset_name': 'YahooAnswersTopics', 'num_labels': 10, 'cache_dir': 'Models'}

The model fabriceyhc/bert-base-uncased-yahoo_answers_topics is loaded.

In [6]:
train_dataloader, valid_dataloader, test_dataloader = load_data(
    name, batch_size=batch_size, num_workers=num_workers, do_cache=True, seed=seed
)

Loading cached dataset YahooAnswersTopics.
The dataset YahooAnswersTopics is loaded
{'dataset_name': 'YahooAnswersTopics', 'path': 'yahoo_answers_topics', 'config_name': 'yahoo_answers_topics', 'text_column': 'question_title', 'label_column': 'topic', 'cache_dir': 'Datasets/Yahoo', 'task_type': 'classification'}


The dataset YahooAnswersTopics is loaded

{'dataset_name': 'YahooAnswersTopics', 'path': 'yahoo_answers_topics', 'config_name': 'yahoo_answers_topics', 'text_column': 'question_title', 'label_column': 'topic', 'cache_dir': 'Datasets/Yahoo', 'task_type': 'classification'}

In [7]:
positive_embeddings, negative_embeddings = make_example(
    model, model_config, data_loader=valid_dataloader,
    example_num=750, emb_num=3, class_num=10, true_ratio=0.5,
    step_epsilon=0.01, max_epsilon=10.0
)

positive num :  375
negative num :  375


Calculating all epsilons:   0%|                                                                               …

per_class_positive_example_num :  37
per_class_negative_example_num :  37


Calculating all epsilons:   0%|                                   | 0/10 [00:00<?, ?it/s]

per_class_positive_example_num : 

37

per_class_negative_example_num : 

37

In [8]:
pos_dataloader = DataLoader(positive_embeddings, batch_size=4, shuffle=True, num_workers=16)
neg_dataloader = DataLoader(negative_embeddings, batch_size=4, shuffle=True, num_workers=16)

In [10]:
result = evaluate_model(model, model_config, pos_dataloader)

Evaluating the model:   0%|                             | 0/628 [00:01<?, ?it/s]

Loss: 0.5769
Precision: 0.8392, Recall: 0.8371, F1-Score: 0.8352
              precision    recall  f1-score   support

           0       0.75      0.77      0.76       255
           1       0.93      0.80      0.86       288
           2       0.85      0.81      0.83       270
           3       0.84      0.83      0.84       300
           4       0.87      0.84      0.86       240
           5       0.90      0.81      0.86       240
           6       0.94      0.83      0.88       252
           7       0.73      0.92      0.81       225
           8       0.73      0.86      0.79       225
           9       0.84      0.90      0.87       216

    accuracy                           0.83      2511
   macro avg       0.84      0.84      0.84      2511
weighted avg       0.84      0.83      0.84      2511



In [11]:
batch_size = 16
num_workers = 4
num_samples = 128
ci_ratio = 0.6
seed = 44
include_layers = ["attention", "intermediate", "output"]
exclude_layers = None

In [12]:
for concern in range(num_labels):
    print(f"-------------------{concern}----------------------")
    module = copy.deepcopy(model)
    pos = copy.deepcopy(pos_dataloader)
    neg = copy.deepcopy(neg_dataloader)
    valid = copy.deepcopy(valid_dataloader)
    
    positive_examples = SamplingDataset(
        pos, concern, num_samples, num_labels, True, 4, device=device, resample=False, seed=seed
    )
    negative_examples = SamplingDataset(
        neg, concern, num_samples, num_labels, False, 4, device=device, resample=False, seed=seed
    )
    prune_concern_identification(
        module,
        model_config,
        positive_examples,
        negative_examples,
        include_layers=include_layers,
        exclude_layers=exclude_layers,
        sparsity_ratio=ci_ratio,
    )

    print(f"Evaluate the pruned model {concern}")
    result = evaluate_model(module, model_config, test_dataloader)
    get_sparsity(module)

    similar(model, module, valid, concern, num_samples, num_labels, device=device, seed=seed)

-------------------0----------------------
Evaluate the pruned model 0


Evaluating the model:   0%|                            | 0/1875 [00:00<?, ?it/s]

Loss: 1.4259
Precision: 0.6455, Recall: 0.5863, F1-Score: 0.5904
              precision    recall  f1-score   support

           0       0.50      0.55      0.52      2941
           1       0.81      0.35      0.48      2997
           2       0.80      0.57      0.66      3016
           3       0.53      0.39      0.45      2978
           4       0.73      0.80      0.76      3017
           5       0.96      0.62      0.75      3004
           6       0.33      0.45      0.38      3037
           7       0.43      0.75      0.55      3026
           8       0.55      0.79      0.65      2997
           9       0.81      0.60      0.69      2987

    accuracy                           0.59     30000
   macro avg       0.65      0.59      0.59     30000
weighted avg       0.65      0.59      0.59     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

Evaluating the model:   0%| | 0/1

Loss: 1.4943
Precision: 0.6452, Recall: 0.5773, F1-Score: 0.5820
              precision    recall  f1-score   support

           0       0.48      0.51      0.50      2941
           1       0.78      0.45      0.57      2997
           2       0.80      0.54      0.64      3016
           3       0.55      0.34      0.42      2978
           4       0.75      0.76      0.76      3017
           5       0.96      0.62      0.75      3004
           6       0.51      0.35      0.42      3037
           7       0.36      0.78      0.49      3026
           8       0.46      0.83      0.59      2997
           9       0.81      0.60      0.69      2987

    accuracy                           0.58     30000
   macro avg       0.65      0.58      0.58     30000
weighted avg       0.65      0.58      0.58     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

...

CCA coefficients mean non-concern: (0.6288765199392534, 0.6288765199392534)

Linear CKA concern: 0.7432576856015659

Linear CKA non-concern: 0.732504462617674

Kernel CKA concern: 0.6892221390223978

Kernel CKA non-concern: 0.5399972319022527